# Streamlined CRISPR Screen Analysis Tutorial

This tutorial demonstrates the essential workflow provided by `streamlined_crispr`.
We generate a small synthetic dataset using ``benchmarking/generate_demo_dataset.py``
so that quality control, effect-size estimation, and differential expression can be
executed end-to-end without external downloads.


## Prerequisites

Install the project in editable mode (with the optional `test` extras) to make
the package importable and provide AnnData, NumPy, pandas, and SciPy:

```bash
pip install -e .[test]
```

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))

from benchmarking.generate_demo_dataset import write_demo_dataset
from streamlined_crispr import (
    compute_average_log_expression,
    compute_pseudobulk_expression,
    preview_backed,
    quality_control_summary,
    wald_test,
    wilcoxon_test,
)


## Prepare the demo dataset

Generate the compact `.h5ad` file if it is missing and preview it with `preview_backed` to inspect key metadata without loading the full matrix.


In [ ]:
data_path = Path('../data/demo_benchmark.h5ad')
output_dir = Path('tutorial_outputs')
output_dir.mkdir(exist_ok=True)

if not data_path.exists():
    print(f'Generating demo dataset at {data_path}')
    write_demo_dataset(data_path)

adata_ro = preview_backed(data_path)
adata_ro.file.close()


## Run quality control

Filtering keeps perturbations with adequate representation and removes low-quality cells and genes. When `control_label` or `gene_name_column` are omitted, the helper logs which defaults are used.


In [ ]:
qc_result = quality_control_summary(
    data_path,
    min_genes=5,
    min_cells_per_perturbation=5,
    min_cells_per_gene=5,
    perturbation_column='perturbation',
    output_dir=output_dir,
    data_name='tutorial',
)
qc_result.filtered_path


In [ ]:
qc_summary = {
    'total_cells': qc_result.cell_mask.size,
    'kept_cells': int(qc_result.cell_mask.sum()),
    'total_genes': qc_result.gene_mask.size,
    'kept_genes': int(qc_result.gene_mask.sum()),
}
qc_summary


## Compute effect sizes

Both effect-size estimators stream counts from disk and persist their outputs for inspection. They reuse the inferred control label and gene identifiers gathered during QC.


In [ ]:
avg_effects = compute_average_log_expression(
    qc_result.filtered_path,
    perturbation_column='perturbation',
    output_dir=output_dir,
    data_name='tutorial_avg',
)
avg_effects.head()


In [ ]:
pseudo_effects = compute_pseudobulk_expression(
    qc_result.filtered_path,
    perturbation_column='perturbation',
    output_dir=output_dir,
    data_name='tutorial_pseudobulk',
)
pseudo_effects.head()


## Differential expression

Run both the Wald and Wilcoxon tests. Each call writes an AnnData file that contains the test statistics and automatically reuses the inferred control label.


In [ ]:
wald_results = wald_test(
    qc_result.filtered_path,
    perturbation_column='perturbation',
    output_dir=output_dir,
    data_name='tutorial_wald',
)
list(wald_results.keys())


In [ ]:
wilcoxon_results = wilcoxon_test(
    qc_result.filtered_path,
    perturbation_column='perturbation',
    output_dir=output_dir,
    data_name='tutorial_wilcoxon',
)
list(wilcoxon_results.groups)


## Inspect generated files

Every stage creates `.h5ad` artifacts for reproducibility and downstream analysis.


In [ ]:
sorted(path.name for path in output_dir.glob('*.h5ad'))
